# 🚗 Car Description — End-to-End Fine-Tuning & Serving Pipeline

**Goal**: Fine-tune **Qwen2.5-3B-Instruct** to extract structured JSON car details from free-text
car-listing descriptions, then serve the adapter with **vLLM** and load-test the endpoint.

### Pipeline Overview

| # | Stage | Description |
|---|-------|-------------|
| 0 | GPU Check | Verify GPU availability |
| 1 | Configuration | Paths, model IDs, hyperparameters |
| 2 | Environment | Install pinned dependencies + LLaMA-Factory |
| 3 | Credentials | HF / W&B login via Colab Secrets |
| 4 | Teacher Server | Install & start Ollama with `qwen2.5:7b` |
| 5 | Schema & Prompts | Pydantic schema + extraction prompt |
| 6 | Dataset | Load `fashxp/cars-description` from HF |
| 7 | Distillation | Label data with the 7B teacher model |
| 8 | Validate & Split | Schema validation + train/val split |
| 9 | Training | QLoRA fine-tune with LLaMA-Factory |
| 10 | Evaluation | Per-field exact-match on held-out set |
| 11 | Publish | Push dataset + LoRA adapter to HF Hub |
| 12 | vLLM Serving | Serve the adapter with vLLM |
| 13 | Load Testing | Headless Locust load test |
| 14 | Cleanup | Stop servers |

**Runtime**: Colab T4 GPU (15 GB VRAM). Run cells **top-to-bottom, in order, once per session.**

> ⚠️ **Before running**: Set `HF_TOKEN` (write scope) and optionally `WANDB_API_KEY` in
> **Colab → Secrets** (🔑 icon in the left sidebar).

## 0 · GPU Check

In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected — go to Runtime → Change runtime type → select a GPU, "
        "then re-run from the top."
    )

props = torch.cuda.get_device_properties(0)
print(f"✅ GPU: {props.name} | VRAM: {props.total_memory / 1e9:.2f} GB")

## 1 · Configuration

All paths, model IDs, and run parameters live here — change **this cell only** when adapting the
notebook to a new dataset, model, or machine.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
from os.path import join

# Reduces CUDA OOM caused by memory fragmentation — must be set before any CUDA allocation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ─── Paths (Google Drive-backed, survives Colab disconnects) ─────────────────
DATA_DIR             = os.environ.get("LLM_FT_DATA_DIR", "/content/drive/MyDrive/llm-finetuning")
LLAMA_FACTORY_DIR    = os.environ.get("LLAMA_FACTORY_DIR", "/content/LLaMA-Factory")

DISTILLED_DATA_PATH  = join(DATA_DIR, "datasets", "distilled", "car_details_extraction.jsonl")
LLAMAFACTORY_DATA_DIR = join(DATA_DIR, "datasets", "llamafactory-finetune-data")
MODEL_OUTPUT_DIR     = join(DATA_DIR, "models", "car-description-lora")

# ─── Hugging Face targets ────────────────────────────────────────────────────
HF_DATASET_ID             = "fashxp/cars-description"
HF_DATASET_SPLIT          = "train"
HF_PUBLISH_DATASET_REPO   = os.environ.get("HF_DATASET_REPO",  "your-username/car-description-extraction")
HF_PUBLISH_MODEL_REPO     = os.environ.get("HF_MODEL_REPO",    "your-username/car-description-lora")
HF_PUBLISH_PRIVATE         = True

# ─── Models ──────────────────────────────────────────────────────────────────
TEACHER_MODEL_ID = "qwen2.5:7b"                # Ollama tag — labels training data only
BASE_MODEL_ID    = "Qwen/Qwen2.5-3B-Instruct"  # HF model id — the one that gets fine-tuned
ADAPTER_NAME     = "car-lora"                   # vLLM LoRA routing name
ADAPTER_REPO     = os.environ.get("HF_MODEL_REPO", "MinaSaher78/car-description-lora")
OLLAMA_HOST      = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
VLLM_HOST        = "http://localhost:8000"

# ─── Run params ──────────────────────────────────────────────────────────────
RANDOM_SEED        = 101
TRAIN_SPLIT_RATIO  = 0.9
DATA_FRACTION      = 0.5    # fraction of raw dataset to use (after shuffling)

# ─── Create directories ─────────────────────────────────────────────────────
for _d in [os.path.dirname(DISTILLED_DATA_PATH), LLAMAFACTORY_DATA_DIR, MODEL_OUTPUT_DIR]:
    os.makedirs(_d, exist_ok=True)

print(f"Data dir:      {DATA_DIR}")
print(f"Distilled to:  {DISTILLED_DATA_PATH}")
print(f"Base model:    {BASE_MODEL_ID}")
print(f"Teacher model: {TEACHER_MODEL_ID}")

## 2 · Environment Setup

> **Run this section once per session.** Do NOT re-run after the kernel restart checkpoint.

Pinned versions prevent the `transformers` / `peft` / LLaMA-Factory mismatch that causes import
errors on Colab. The install order matters:
1. Install standalone packages first.
2. Clone & install LLaMA-Factory (which pulls its own transformers/peft).
3. Force-pin `transformers==4.57.6` + `peft==0.18.1` **after** LLaMA-Factory to override whatever it resolved.

In [ ]:
# ── Step 1: Standalone packages ──────────────────────────────────────────────
!pip install -q     "datasets==3.2.0"     "optimum==1.24.0"     "openai==1.61.0"     "json-repair==0.29.1"     "faker==35.2.0"     "bitsandbytes"     "wandb"     "pyyaml"     "locust"

print("✅ Step 1 complete: standalone packages installed.")

In [ ]:
# ── Step 2: Clone & install LLaMA-Factory ────────────────────────────────────
import os

if not os.path.isdir(LLAMA_FACTORY_DIR):
    !git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git {LLAMA_FACTORY_DIR}
else:
    print(f"LLaMA-Factory already cloned at {LLAMA_FACTORY_DIR}")

!pip install -q -e "{LLAMA_FACTORY_DIR}[torch,metrics]"

print("✅ Step 2 complete: LLaMA-Factory installed.")

In [ ]:
# ── Step 3: Force-pin the exact transformers/peft pairing ────────────────────
# LLaMA-Factory's resolver may pull a newer transformers; this overrides it.
# transformers==4.57.6 + peft==0.18.1 is the confirmed-working combination.
!pip install -q "transformers==4.57.6" "peft==0.18.1"

print("✅ Step 3 complete: transformers/peft pinned.")

In [ ]:
# ── Step 4: Verify versions ──────────────────────────────────────────────────
import transformers, peft
from peft.utils.save_and_load import _maybe_shard_state_dict_for_tp  # fails if versions are wrong

print(f"transformers: {transformers.__version__} (need 4.57.6)")
print(f"peft:         {peft.__version__} (need 0.18.1)")

assert transformers.__version__ == "4.57.6", f"Wrong transformers version: {transformers.__version__}"
assert peft.__version__ == "0.18.1", f"Wrong peft version: {peft.__version__}"

print("✅ All versions verified — environment is ready.")

## 3 · Credentials

Tokens are loaded from **Colab Secrets** (🔑 icon in the left sidebar).
- `HF_TOKEN`: Required (write scope) for pushing to the Hub.
- `WANDB_API_KEY`: Optional — for Weights & Biases experiment tracking.

In [ ]:
import os
import wandb

# ── Load tokens from Colab Secrets ───────────────────────────────────────────
try:
    from google.colab import userdata
    hf_token  = userdata.get("HF_TOKEN")
    wandb_key = userdata.get("WANDB_API_KEY")
except (ImportError, Exception):
    hf_token  = os.environ.get("HF_TOKEN")
    wandb_key = os.environ.get("WANDB_API_KEY")

# ── HuggingFace login ────────────────────────────────────────────────────────
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    !huggingface-cli login --token $hf_token
else:
    print("⚠️  HF_TOKEN not set — gated model downloads and Hub publishing will fail.")
    print("   Set it in Colab → Secrets (🔑) with write scope.")

# ── Weights & Biases login ───────────────────────────────────────────────────
if wandb_key:
    wandb.login(key=wandb_key)
    print("✅ W&B login successful.")
else:
    print("ℹ️  WANDB_API_KEY not set — training will use report_to='none'.")

## 4 · Teacher Model Server (Ollama)

The teacher model (`qwen2.5:7b`) is used **only** to generate labeled training data — it is never
fine-tuned itself. We run it locally on the Colab GPU via Ollama.

In [ ]:
import subprocess, time, shutil, requests

# Ollama needs zstd to unpack itself — not preinstalled on Colab.
!apt-get -qq update && apt-get -qq install -y zstd

# ── Find or install Ollama ───────────────────────────────────────────────────
def find_ollama():
    for path in [shutil.which("ollama"), "/usr/local/bin/ollama", "/usr/bin/ollama"]:
        if path and os.path.exists(path):
            return path
    return None

ollama_bin = find_ollama()

if ollama_bin is None:
    print("Ollama not found — installing...")
    result = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Ollama install failed with code {result.returncode}")
    os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]
    ollama_bin = find_ollama()

if ollama_bin is None:
    raise FileNotFoundError("Ollama still not found after install — check the install output above.")

print(f"Using ollama binary: {ollama_bin}")

# ── Start the server ─────────────────────────────────────────────────────────
!pkill -9 ollama || true

print("Starting Ollama server...")
with open("ollama.log", "w") as f:
    ollama_process = subprocess.Popen([ollama_bin, "serve"], stdout=f, stderr=f, env=os.environ)

server_ready = False
for _ in range(30):
    try:
        if requests.get(OLLAMA_HOST + "/").status_code == 200:
            server_ready = True
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1.0)

if server_ready:
    print("✅ Ollama server is up.")
else:
    print("❌ Ollama did not respond. Last log lines:")
    print(open("ollama.log").read()[-1000:])

In [ ]:
# ── Pull the teacher model ───────────────────────────────────────────────────
!ollama pull {TEACHER_MODEL_ID}

resp = requests.get(OLLAMA_HOST + "/api/tags")
pulled = [m["name"] for m in resp.json().get("models", [])]
print("Pulled models:", pulled)
assert any(TEACHER_MODEL_ID in m for m in pulled), f"{TEACHER_MODEL_ID} not found in pulled models."
print(f"✅ Teacher model ready: {TEACHER_MODEL_ID}")

## 5 · Pydantic Schema & Prompt Builder

The target output schema for structured car-detail extraction. Every distilled record and every
model prediction is validated against this schema.

In [ ]:
import json
import random
import re
from typing import Literal, Optional
from datetime import datetime

from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
import json_repair

# ── Schema ───────────────────────────────────────────────────────────────────
BodyStyle = Literal[
    "2-door roadster", "convertible", "coupe", "sedan", "suv", "hatchback",
    "wagon", "pickup-truck", "van", "limousine", "not_specified"
]
DriveType = Literal["front-wheel drive", "rear-wheel drive", "all-wheel drive", "not_specified"]
TransmissionType = Literal["manual", "automatic", "semi-automatic", "cvt", "not_specified"]
FuelType = Literal["gasoline", "diesel", "electric", "hybrid", "plug-in hybrid", "hydrogen", "cng", "lpg", "not_specified"]
Condition = Literal["new", "used", "certified pre-owned", "not_specified"]

class CarDetails(BaseModel):
    name: str = Field(..., description="Full model name including manufacturer")
    year: Optional[int] = Field(None, description="Model year, if mentioned")
    body_style: BodyStyle = "not_specified"
    exterior_color: str = "not_specified"
    interior_color: str = "not_specified"
    engine_type: str = "not_specified"
    engine_cylinders: Optional[int] = Field(None, description="Number of cylinders")
    engine_power: str = "not_specified"
    drive_type: DriveType = "not_specified"
    transmission: TransmissionType = "not_specified"
    fuel_type: FuelType = "not_specified"
    mileage: str = "not_specified"
    price: str = "not_specified"
    condition: Condition = "not_specified"
    vin: str = "not_specified"
    features: list[str] = Field(default_factory=list, description="Notable features / options")

# ── Extraction prompt ────────────────────────────────────────────────────────
EXTRACTION_SYSTEM_PROMPT = "\n".join([
    "You are an expert automotive data extractor.",
    "Given a car listing description, extract the structured details into JSON that conforms to the provided Pydantic schema.",
    "Return ONLY the JSON object — no markdown fences, no commentary.",
    "If a field is not mentioned in the text, use 'not_specified' for enum fields or omit optional fields.",
    "Preserve numeric values with their original units exactly as written in the text (e.g. '4,285 m', '110 kw').",
    "Do not generate any introduction or conclusion.",
])

CAR_DETAILS_SCHEMA_JSON = json.dumps(CarDetails.model_json_schema(), ensure_ascii=False)


def build_extraction_messages(description: str) -> list:
    """Chat messages used to prompt a model to extract CarDetails JSON from a car description."""
    return [
        {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "\n".join([
                "## Car Description:",
                description.strip(),
                "",
                "## Pydantic Schema:",
                CAR_DETAILS_SCHEMA_JSON,
                "",
                "## Extracted Details:",
                "```json",
            ]),
        },
    ]


def parse_json(text: str):
    """Parse JSON from model output, with json-repair fallback."""
    text = text.strip()
    # Strip markdown fences if present
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"```\s*$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        try:
            return json_repair.loads(text)
        except Exception:
            return None


# ── Teacher model client ─────────────────────────────────────────────────────
ollama_client = OpenAI(base_url=OLLAMA_HOST + "/v1", api_key="ollama")


def call_teacher_model(messages, retries=3, backoff=5):
    """Call the Ollama teacher model with retry logic."""
    for attempt in range(retries):
        try:
            return ollama_client.chat.completions.create(
                model=TEACHER_MODEL_ID,
                messages=messages,
                temperature=0.3,
                max_tokens=2000,
            )
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"  Attempt {attempt+1} failed: {e} — retrying in {backoff}s...")
            time.sleep(backoff)

print("✅ Schema, prompts, and teacher client ready.")

## 6 · Load & Preview Dataset

Load the raw car-description dataset from Hugging Face and take a `DATA_FRACTION` sample.

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

ds = load_dataset(HF_DATASET_ID)
print(ds)

full_data = list(ds[HF_DATASET_SPLIT])
random.seed(RANDOM_SEED)
random.shuffle(full_data)
raw_data = full_data[:int(len(full_data) * DATA_FRACTION)]

print(f"\nUsing {len(raw_data)}/{len(full_data)} samples ({DATA_FRACTION:.0%})")
print(f"\nSample description:\n{raw_data[0]['Description'][:500]}...")

In [ ]:
# ── Quick base-model sanity check ────────────────────────────────────────────
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, device_map="auto")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)


def generate_with_model(model, tokenizer, messages, max_new_tokens=1024):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    gen_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    gen_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, gen_ids)]
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]


sample_msgs = build_extraction_messages(raw_data[0]["Description"])
base_response = generate_with_model(base_model, base_tokenizer, sample_msgs, max_new_tokens=500)
print("Base model (before fine-tuning) response:")
print(base_response[:1000])

In [ ]:
# Free GPU memory — the base model is not needed during distillation.
def free_gpu_memory(*var_names):
    import gc
    g = globals()
    for name in var_names:
        if name in g:
            del g[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

free_gpu_memory("base_model")
print("✅ Base model freed from GPU.")

## 7 · Data Distillation

Use the 7B teacher model (Ollama) to label each car description with structured JSON.
This loop is **resumable** — it appends to the output file and skips already-processed IDs.

In [ ]:
# ── Estimate time ────────────────────────────────────────────────────────────
N_ESTIMATE = 30

start = datetime.now()
for i in range(min(N_ESTIMATE, len(raw_data))):
    try:
        call_teacher_model(build_extraction_messages(raw_data[i]["Description"]))
    except Exception:
        pass

elapsed = (datetime.now() - start).total_seconds()
per_sample = elapsed / min(N_ESTIMATE, len(raw_data))
estimated_total = per_sample * len(raw_data) / 60

print(f"⏱  {per_sample:.1f}s per sample → ~{estimated_total:.0f} min for {len(raw_data)} samples")

In [ ]:
# ── Distillation loop (resumable) ────────────────────────────────────────────
def load_processed_ids(path):
    if not os.path.exists(path):
        return set()
    ids = set()
    with open(path, encoding="utf8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                ids.add(json.loads(line)["id"])
            except Exception:
                continue
    return ids

processed_ids = load_processed_ids(DISTILLED_DATA_PATH)
print(f"{len(processed_ids)} samples already distilled." if processed_ids else "Starting fresh distillation.")

prompt_tokens = 0
completion_tokens = 0
failures = 0

with open(DISTILLED_DATA_PATH, "a", encoding="utf8") as dest:
    for ix, car in enumerate(tqdm(raw_data, desc="Distilling")):
        if ix in processed_ids:
            continue

        try:
            response = call_teacher_model(build_extraction_messages(car["Description"]))
        except Exception as e:
            print(f"  Request failed for index {ix}: {e}")
            failures += 1
            continue

        if response.choices[0].finish_reason != "stop":
            if response.usage:
                prompt_tokens += response.usage.prompt_tokens
            failures += 1
            continue

        parsed = parse_json(response.choices[0].message.content)
        if not parsed:
            failures += 1
            continue

        dest.write(json.dumps({
            "id": ix,
            "description": car["Description"].strip(),
            "task": "Extract the car details into a JSON.",
            "output_schema": CAR_DETAILS_SCHEMA_JSON,
            "response": parsed,
        }, ensure_ascii=False, default=str) + "\n")
        dest.flush()

        if response.usage:
            prompt_tokens += response.usage.prompt_tokens
            completion_tokens += response.usage.completion_tokens

print(f"\n✅ Done — Prompt tokens: {prompt_tokens} | Completion tokens: {completion_tokens} | Failures: {failures}")

## 8 · Validate & Split Data

Validate distilled records against the Pydantic schema, then format for LLaMA-Factory and
split into train/val sets.

In [ ]:
# ── Load & validate ──────────────────────────────────────────────────────────
with open(DISTILLED_DATA_PATH, encoding="utf8") as f:
    distilled_records = [json.loads(line) for line in f if line.strip()]

valid_records = []
invalid_count = 0

for rec in distilled_records:
    try:
        CarDetails(**rec["response"])
        valid_records.append(rec)
    except Exception:
        invalid_count += 1

print(f"Total distilled: {len(distilled_records)} | Valid: {len(valid_records)} | Invalid: {invalid_count}")

In [ ]:
# ── Convert to LLaMA-Factory format & split ──────────────────────────────────
def to_llamafactory_record(rec):
    return {
        "instruction": rec["task"],
        "input": "\n".join([
            "## Car Description:",
            rec["description"],
            "",
            "## Pydantic Schema:",
            rec["output_schema"],
        ]),
        "output": json.dumps(rec["response"], ensure_ascii=False),
        "system": EXTRACTION_SYSTEM_PROMPT,
        "history": [],
    }

llm_data = [to_llamafactory_record(r) for r in valid_records]
random.Random(RANDOM_SEED).shuffle(llm_data)

split_ix = int(len(llm_data) * TRAIN_SPLIT_RATIO)
train_ds = llm_data[:split_ix]
eval_ds  = llm_data[split_ix:]

train_path = join(LLAMAFACTORY_DATA_DIR, "train.json")
val_path   = join(LLAMAFACTORY_DATA_DIR, "val.json")

with open(train_path, "w", encoding="utf8") as f:
    json.dump(train_ds, f, ensure_ascii=False, default=str)
with open(val_path, "w", encoding="utf8") as f:
    json.dump(eval_ds, f, ensure_ascii=False, default=str)

print(f"Train: {len(train_ds)} → {train_path}")
print(f"Val:   {len(eval_ds)} → {val_path}")

In [ ]:
# ── Register in LLaMA-Factory's dataset_info.json ────────────────────────────
dataset_info_path = join(LLAMA_FACTORY_DIR, "data", "dataset_info.json")

with open(dataset_info_path, encoding="utf8") as f:
    dataset_info = json.load(f)

dataset_info["car_description_train"] = {"file_name": os.path.abspath(train_path)}
dataset_info["car_description_val"]   = {"file_name": os.path.abspath(val_path)}

with open(dataset_info_path, "w", encoding="utf8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"✅ Registered datasets in {dataset_info_path}")

## 9 · QLoRA Training

Fine-tune `Qwen2.5-3B-Instruct` with QLoRA (4-bit quantized base, LoRA adapters in fp16/bf16).
Training auto-resumes from the latest checkpoint if a previous run was interrupted.

In [ ]:
# Free the Ollama teacher model — we don't need it anymore, and training needs the GPU.
free_gpu_memory()
!ollama stop {TEACHER_MODEL_ID} 2>/dev/null || true
print("✅ Teacher model stopped, GPU freed for training.")

In [ ]:
import yaml, shutil

# ── Detect checkpoint for resume ─────────────────────────────────────────────
def find_latest_checkpoint(output_dir):
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if re.match(r"checkpoint-\d+", d)]
    if not checkpoints:
        return None
    latest = max(checkpoints, key=lambda d: int(d.split("-")[1]))
    return os.path.join(output_dir, latest)

resume_checkpoint = find_latest_checkpoint(MODEL_OUTPUT_DIR)

if resume_checkpoint:
    backup = f"{MODEL_OUTPUT_DIR}_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    shutil.copytree(MODEL_OUTPUT_DIR, backup)
    print(f"📂 Resuming from: {resume_checkpoint}")
    print(f"   Backed up to:  {backup}")
else:
    print("🆕 No checkpoint found — starting training from scratch.")

In [ ]:
# ── Write training config ────────────────────────────────────────────────────
lora_config = {
    "model_name_or_path": BASE_MODEL_ID,
    "trust_remote_code": True,

    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_rank": 32,
    "lora_target": "all",

    # QLoRA: 4-bit quantized base model
    "quantization_bit": 4,
    "quantization_method": "bitsandbytes",
    "double_quantization": True,

    "dataset": "car_description_train",
    "eval_dataset": "car_description_val",
    "template": "qwen",
    "cutoff_len": 1024,
    "overwrite_cache": True,
    "preprocessing_num_workers": 16,

    "output_dir": MODEL_OUTPUT_DIR,
    "resume_from_checkpoint": resume_checkpoint,
    "logging_steps": 10,
    "save_steps": 10,
    "plot_loss": True,

    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "gradient_checkpointing": True,
    "optim": "paged_adamw_8bit",
    "learning_rate": 1.0e-4,
    "num_train_epochs": 3.0,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "bf16": True,
    "ddp_timeout": 180000000,

    "per_device_eval_batch_size": 1,
    "eval_strategy": "steps",
    "eval_steps": 20,

    "report_to": "wandb" if wandb_key else "none",
    "run_name": "car-description-lora",
}

lora_config_path = join(LLAMA_FACTORY_DIR, "examples", "train_lora", "car_description.yaml")
os.makedirs(os.path.dirname(lora_config_path), exist_ok=True)
with open(lora_config_path, "w", encoding="utf8") as f:
    yaml.dump(lora_config, f, sort_keys=False)

print(f"✅ Training config written to {lora_config_path}")

In [ ]:
# ── Launch training ──────────────────────────────────────────────────────────
!cd {LLAMA_FACTORY_DIR} && llamafactory-cli train {os.path.abspath(lora_config_path)}

## 10 · Evaluation

Reload the base model with the trained LoRA adapter and evaluate per-field exact-match accuracy
on the held-out validation set.

In [ ]:
from peft import PeftModel

# ── Reload base model + apply adapter ────────────────────────────────────────
if "base_model" not in dir() or base_model is None:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, device_map="auto", torch_dtype=torch.float16
    )

# Find the adapter path (last checkpoint or final output)
adapter_path = find_latest_checkpoint(MODEL_OUTPUT_DIR) or MODEL_OUTPUT_DIR
base_model = PeftModel.from_pretrained(base_model, adapter_path)
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

print(f"✅ Loaded adapter from: {adapter_path}")

In [ ]:
# ── Generator with Chinese-token ban ─────────────────────────────────────────
class Generator:
    """Chat generation helper that bans Chinese-character tokens (Qwen2.5 quirk)."""

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.mask = None

    def generate(self, messages, max_new_tokens=2000, temperature=0.1):
        def logits_processor(token_ids, logits):
            if self.mask is None:
                vocab_ids = torch.arange(logits.size(-1))
                decoded = self.tokenizer.batch_decode(vocab_ids.unsqueeze(1), skip_special_tokens=True)
                self.mask = torch.tensor([
                    any(0x4E00 <= ord(c) <= 0x9FFF or 0x3400 <= ord(c) <= 0x4DBF
                        or 0xF900 <= ord(c) <= 0xFAFF for c in tok)
                    for tok in decoded
                ])
            logits[:, self.mask] = -float("inf")
            return logits

        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        gen_ids = self.model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, logits_processor=[logits_processor],
        )
        gen_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, gen_ids)]
        return self.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]

finetuned_llm = Generator(base_model, base_tokenizer)

# Quick sanity check
sample = raw_data[len(train_ds) if "train_ds" in dir() else 0]
print("Fine-tuned model response:")
print(finetuned_llm.generate(build_extraction_messages(sample["Description"])))

In [ ]:
# ── Per-field exact-match evaluation ─────────────────────────────────────────
field_correct = {}
field_total = {}
parse_failures = 0

for record in tqdm(eval_ds, desc="Evaluating"):
    target = json.loads(record["output"])

    # Extract original description from the record's input
    inp = record["input"]
    start = inp.find("## Car Description:\n") + len("## Car Description:\n")
    end = inp.find("\n\n## Pydantic Schema:")
    description = inp[start:end].strip()

    raw_resp = finetuned_llm.generate(build_extraction_messages(description))
    predicted = parse_json(raw_resp)

    if predicted is None:
        parse_failures += 1
        continue

    for key, target_val in target.items():
        field_total[key] = field_total.get(key, 0) + 1
        if str(predicted.get(key, "")).strip().lower() == str(target_val).strip().lower():
            field_correct[key] = field_correct.get(key, 0) + 1

print(f"\nParse failures: {parse_failures} / {len(eval_ds)}")
print(f"{'Field':<20}{'Exact match'}")
print("-" * 35)
for key in sorted(field_total):
    rate = field_correct.get(key, 0) / field_total[key]
    print(f"{key:<20}{rate:.1%} ({field_correct.get(key, 0)}/{field_total[key]})")

overall = sum(field_correct.values()) / max(sum(field_total.values()), 1)
print(f"\n{'Overall':<20}{overall:.1%}")

## 11 · Publish to Hugging Face Hub

Push the distilled dataset and the LoRA adapter to the Hub. The `your-username` placeholder
is automatically resolved from your HF login.

In [ ]:
from huggingface_hub import HfApi, create_repo, whoami

assert hf_token, "HF_TOKEN not set — required to publish (needs write scope)."
api = HfApi()

# ── Resolve username ─────────────────────────────────────────────────────────
username = whoami(token=hf_token)["name"]

repo_id_dataset = HF_PUBLISH_DATASET_REPO.replace("your-username", username)
repo_id_model   = HF_PUBLISH_MODEL_REPO.replace("your-username", username)

print(f"Dataset repo: {repo_id_dataset}")
print(f"Model repo:   {repo_id_model}")

In [ ]:
# ── Push dataset ─────────────────────────────────────────────────────────────
create_repo(repo_id_dataset, repo_type="dataset", private=HF_PUBLISH_PRIVATE, exist_ok=True)

api.upload_file(path_or_fileobj=DISTILLED_DATA_PATH, path_in_repo="car_details_extraction.jsonl",
                repo_id=repo_id_dataset, repo_type="dataset")
api.upload_file(path_or_fileobj=train_path, path_in_repo="train.json",
                repo_id=repo_id_dataset, repo_type="dataset")
api.upload_file(path_or_fileobj=val_path, path_in_repo="val.json",
                repo_id=repo_id_dataset, repo_type="dataset")

print(f"✅ Dataset pushed to https://huggingface.co/datasets/{repo_id_dataset}")

In [ ]:
# ── Push LoRA adapter ────────────────────────────────────────────────────────
create_repo(repo_id_model, private=HF_PUBLISH_PRIVATE, exist_ok=True)

api.upload_folder(folder_path=adapter_path, repo_id=repo_id_model)

# Model card
model_card = f"""---
base_model: {BASE_MODEL_ID}
library_name: peft
tags:
- lora
- qlora
- car-description
- json-extraction
---

# {repo_id_model.split('/')[-1]}

LoRA adapter fine-tuned on **{BASE_MODEL_ID}** to extract structured JSON car details from
free-text car listing descriptions.

## Training Data

Distilled from [`{HF_DATASET_ID}`](https://huggingface.co/datasets/{HF_DATASET_ID}) using
`{TEACHER_MODEL_ID}` as teacher. Published: [`{repo_id_dataset}`](https://huggingface.co/datasets/{repo_id_dataset}).

## Output Schema

```json
{CAR_DETAILS_SCHEMA_JSON}
```

## Training Config

- Method: QLoRA (4-bit, bitsandbytes)
- LoRA rank: {lora_config['lora_rank']}, target: {lora_config['lora_target']}
- Epochs: {lora_config['num_train_epochs']}, LR: {lora_config['learning_rate']}

## Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("{BASE_MODEL_ID}")
model = PeftModel.from_pretrained(model, "{repo_id_model}")
tokenizer = AutoTokenizer.from_pretrained("{BASE_MODEL_ID}")
```
"""

api.upload_file(path_or_fileobj=model_card.encode("utf-8"), path_in_repo="README.md",
                repo_id=repo_id_model)
print(f"✅ Model pushed to https://huggingface.co/{repo_id_model}")

In [ ]:
# ── (Optional) Merge & export full model ─────────────────────────────────────
merged_dir = join(DATA_DIR, "models", "car-description-merged")

export_config = {
    "model_name_or_path": BASE_MODEL_ID,
    "adapter_name_or_path": adapter_path,
    "template": "qwen",
    "finetuning_type": "lora",
    "export_dir": merged_dir,
    "export_size": 2,
    "export_device": "cpu",
    "export_legacy_format": False,
}
export_path = join(LLAMA_FACTORY_DIR, "examples", "merge_lora", "car_description_merge.yaml")
os.makedirs(os.path.dirname(export_path), exist_ok=True)
with open(export_path, "w", encoding="utf8") as f:
    yaml.dump(export_config, f, sort_keys=False)

!cd {LLAMA_FACTORY_DIR} && llamafactory-cli export {os.path.abspath(export_path)}

# Push merged
HF_PUBLISH_MERGED_REPO = repo_id_model + "-merged"
create_repo(HF_PUBLISH_MERGED_REPO, private=HF_PUBLISH_PRIVATE, exist_ok=True)
api.upload_folder(folder_path=merged_dir, repo_id=HF_PUBLISH_MERGED_REPO)
print(f"✅ Merged model pushed to https://huggingface.co/{HF_PUBLISH_MERGED_REPO}")

## 12 · vLLM Serving

Serve the fine-tuned LoRA adapter with vLLM's OpenAI-compatible endpoint.

> **Note**: T4 GPUs lack Flash Attention 2 support. We use `--enforce-eager` and limit
> `--max-model-len` to 4096 to fit within 15 GB VRAM with LoRA enabled.

In [ ]:
# ── Free GPU before vLLM ─────────────────────────────────────────────────────
free_gpu_memory("base_model", "finetuned_llm")

# ── Install vLLM ─────────────────────────────────────────────────────────────
!pip install -q vllm
!pip install -qU huggingface_hub

# Fix torchaudio CUDA version to match what vLLM's torch expects
import subprocess
cuda_ver = subprocess.check_output(
    "python -c \"import torch; print(torch.version.cuda.replace('.',''))\"",
    shell=True).decode().strip()
print(f"PyTorch CUDA version: {cuda_ver}")
!pip install -q --no-cache-dir --force-reinstall torchaudio --index-url https://download.pytorch.org/whl/cu{cuda_ver}

print("✅ vLLM installed.")

In [ ]:
# ── Download adapter from Hub ────────────────────────────────────────────────
from huggingface_hub import snapshot_download

LOCAL_ADAPTER_DIR = "/content/car-description-lora"
os.makedirs(LOCAL_ADAPTER_DIR, exist_ok=True)

adapter_local = snapshot_download(repo_id=ADAPTER_REPO, local_dir=LOCAL_ADAPTER_DIR)
print(f"Adapter downloaded to: {adapter_local}")
print("Contents:", os.listdir(adapter_local))

assert any(f.startswith("adapter_model") for f in os.listdir(adapter_local)), \
    "No adapter_model.* file found — check ADAPTER_REPO."


In [ ]:
# ── Launch vLLM server ───────────────────────────────────────────────────────
import subprocess, time, requests

# Kill any existing vLLM processes
print("Stopping any running vLLM servers...")
subprocess.run("pkill -9 -f 'vllm serve'", shell=True)
time.sleep(2.0)

print("Launching vLLM server...")
with open("nohup.out", "w") as f:
    vllm_process = subprocess.Popen(
        ["vllm", "serve", BASE_MODEL_ID,
         "--dtype", "half",
         "--max-model-len", "4096",
         "--gpu-memory-utilization", "0.8",
         "--max_lora_rank", "64",
         "--enable-lora",
         "--lora-modules", f"{ADAPTER_NAME}={adapter_local}",
         "--enforce-eager"],
        stdout=f, stderr=f,
    )

# Poll for readiness (up to 90s — T4 model loading can be slow)
server_ready = False
print("Waiting for server to start (up to 90s)...")
for i in range(45):
    try:
        if requests.get(f"{VLLM_HOST}/health").status_code == 200:
            server_ready = True
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2.0)

if server_ready:
    print("✅ vLLM server is up and healthy!")
else:
    print("❌ Server did not start. Last 30 log lines:")
    with open("nohup.out", "r", errors="ignore") as f:
        print("".join(f.readlines()[-30:]))

In [ ]:
# ── Inference sanity check ───────────────────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

sample_description = (
    "The Mercedes-Benz W113 is a luxurious 2-door roadster, finished in silver. "
    "This 1965 classic is powered by a 6-cylinder engine producing 110 kW, with rear-wheel drive."
)

prompt = tokenizer.apply_chat_template(
    build_extraction_messages(sample_description),
    tokenize=False, add_generation_prompt=True,
)

try:
    resp = requests.post(f"{VLLM_HOST}/v1/completions", json={
        "model": ADAPTER_NAME,
        "prompt": prompt,
        "max_tokens": 1000,
        "temperature": 0.3,
    })
    result = resp.json()
    print("✅ vLLM inference response:")
    print(json.dumps(result, indent=2))
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to vLLM server — ensure Section 12 launched successfully.")
    print("   Check logs with: !tail -n 30 nohup.out")

## 13 · Load Testing (Locust)

Run a headless Locust load test against the vLLM endpoint: 20 concurrent users, 60-second run.

In [ ]:
%%writefile /content/locust_test.py

import random
import json
from locust import HttpUser, task, between
from faker import Faker

fake = Faker("en_US")


class CompletionLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def post_completion(self):
        message = {
            "model": "car-lora",
            "prompt": fake.text(max_nb_chars=random.randint(150, 200)),
            "max_tokens": 512,
            "temperature": 0.3,
        }
        response = self.client.post("/v1/completions", json=message)
        if response.status_code == 200:
            with open("/content/vllm_tokens.txt", "a") as dest:
                dest.write(json.dumps({
                    "prompt": message["prompt"],
                    "response": response.json()["choices"][0]["text"],
                }, ensure_ascii=False) + "\n")

In [ ]:
# ── Run load test ────────────────────────────────────────────────────────────
!locust --headless -f /content/locust_test.py --host={VLLM_HOST} -u 20 -r 1 -t "60s" --html=/content/locust_results.html

In [ ]:
# ── Token throughput stats ───────────────────────────────────────────────────
tokens_file = "/content/vllm_tokens.txt"
if os.path.exists(tokens_file):
    vllm_tokens = [json.loads(l.strip()) for l in open(tokens_file) if l.strip()]
    total_in  = sum(len(tokenizer.encode(r["prompt"]))   for r in vllm_tokens)
    total_out = sum(len(tokenizer.encode(r["response"])) for r in vllm_tokens)

    print(f"Successful requests:  {len(vllm_tokens)}")
    print(f"Total input tokens:   {total_in:,}")
    print(f"Total output tokens:  {total_out:,}")
    print(f"Output tokens/sec:    {total_out / 60:.1f}")
else:
    print("No token data found — load test may have failed.")

In [ ]:
# ── Download results ─────────────────────────────────────────────────────────
from google.colab import files
files.download("/content/locust_results.html")

## 14 · Cleanup

Stop background servers and free resources.

In [ ]:
!pkill -9 -f "vllm serve" || true
!pkill -9 ollama || true

print("✅ All servers stopped.")
print("\n🎉 Pipeline complete! Your fine-tuned adapter is published to the Hugging Face Hub.")
